# 11 — OpenCV Video & Real-Time Computer Vision

Each major concept uses **Intuition → Straight definition → One example**.

## 1. What is a video?
**Intuition:** A video is many images shown one after another.

**Definition:** OpenCV reads a video as a sequence of frames; each frame is a NumPy image array.

**Example:** 30 FPS means about 30 frames per second.

## 2. `VideoCapture`
**Intuition:** It is the connection between your program and a camera/video file.

**Definition:** `cv2.VideoCapture(source)` opens a camera or video source.

**Example:** `0` = default webcam; `'video.mp4'` = video file.

In [1]:
import cv2

cap = cv2.VideoCapture('opencv_sample_video.mp4')
print(cap.isOpened())

True


## 3. Read one frame
**Intuition:** `read()` asks for the next image.

**Definition:** `read()` returns `(success, frame)`.

**Example:**

In [2]:
ret, frame = cap.read()

print(ret)

if ret:
    print(frame.shape)

True
(360, 640, 3)


## 4. Read every frame
**Intuition:** Keep asking for frames until the video ends.

**Definition:** A video loop repeatedly calls `read()` and processes each frame.

**Example:**

In [3]:
cap = cv2.VideoCapture('opencv_sample_video.mp4')

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.imshow('Video', frame)

    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## 5. `waitKey()`
**Intuition:** OpenCV needs a small wait to refresh the window and receive keyboard input.

**Definition:** `cv2.waitKey(delay)` waits approximately `delay` milliseconds.

**Example:** `cv2.waitKey(1)` is common in real-time loops.

## 6. Webcam
**Intuition:** The camera continuously supplies new frames.

**Definition:** `cv2.VideoCapture(0)` opens the default camera.

**Example:**

In [4]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.imshow('Webcam', frame)

    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## 7. Video properties
**Intuition:** A video has width, height, FPS, and frame count.

**Definition:** `cap.get()` reads capture properties.

**Example:**

In [5]:
cap = cv2.VideoCapture('opencv_sample_video.mp4')

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print('Width:', width)
print('Height:', height)
print('FPS:', fps)
print('Frames:', frame_count)

cap.release()

Width: 640
Height: 360
FPS: 30.0
Frames: 240


## 8. Jump to a frame
**Intuition:** You can move directly to a chosen frame.

**Definition:** `CAP_PROP_POS_FRAMES` controls the current frame position.

**Example:**

In [6]:
cap = cv2.VideoCapture('opencv_sample_video.mp4')
cap.set(cv2.CAP_PROP_POS_FRAMES, 100)
ret, frame = cap.read()

if ret:
    cv2.imshow('Frame 100', frame)
    cv2.waitKey(0)

cap.release()
cv2.destroyAllWindows()

## 9. Process every frame
**Intuition:** Real-time CV applies an operation to each incoming image.

**Definition:** The basic pipeline is **capture → process → display/output**.

**Example: grayscale video**

In [7]:
cap = cv2.VideoCapture('opencv_sample_video.mp4')

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cv2.imshow('Gray Video', gray)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## 10. Resize frames
**Intuition:** Smaller frames usually make processing faster.

**Definition:** `cv2.resize()` changes spatial resolution.

**Example:**

In [8]:
cap = cv2.VideoCapture('opencv_sample_video.mp4')

while True:
    ret, frame = cap.read()
    if not ret:
        break


    small = cv2.resize(frame, (640, 480))
    cv2.imshow('Resize', small)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
cap.release()
cv2.destroyAllWindows()

## 11. FPS and real-time speed
**Intuition:** If input arrives at 30 FPS but your algorithm handles only 10 FPS, it cannot keep up.

**Definition:** FPS is frames processed/displayed per second.

If one frame takes `T` seconds:

**FPS = `1 / T`**

Example: `T = 0.05 s` → `20 FPS`.

## 12. Measure processing FPS
**Intuition:** Measure processing time, then invert it.

**Definition:** `time.perf_counter()` is a high-resolution timer for elapsed-time measurement.

**Example:**

In [9]:
import time

cap = cv2.VideoCapture("opencv_sample_video.mp4")

ret, frame = cap.read()

start = time.perf_counter()

_ = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

elapsed = time.perf_counter() - start

fps = 1 / elapsed if elapsed > 0 else 0

print(f'Processing FPS: {fps:.1f}')

Processing FPS: 6934.8


## 13. Display FPS
**Intuition:** Put the current speed directly on the frame.

**Definition:** `cv2.putText()` draws text on an image.

**Example:**

In [10]:
cv2.putText(frame, f'FPS: {fps:.1f}', (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

cv2.imshow("show FPS", frame)

## 14. `VideoWriter`
**Intuition:** `VideoCapture` reads video; `VideoWriter` saves processed frames.

**Definition:** `cv2.VideoWriter()` creates a video output stream.

**Example:**

In [15]:
cap = cv2.VideoCapture('opencv_sample_video.mp4')
fps = cap.get(cv2.CAP_PROP_FPS) or 30
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, fps, (width, height))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    output = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    out.write(output)

cap.release()
out.release()

## 15. Camera resolution
**Intuition:** You can request a camera resolution, but the camera may choose the closest supported mode.

**Definition:** `cap.set(property, value)` requests a capture property.

**Example:**

In [16]:
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
print(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

1280.0
720.0


## 16. Real-time CV pipeline
**Intuition:** This is the pattern behind webcam filters, detection, tracking, and YOLO.

**Definition:** Capture → preprocess → CV/AI → annotate → display/save/action.

**Example:**

In [18]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    small = cv2.resize(frame, (640, 480))
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    cv2.imshow('Real-Time CV', gray)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## 17. Complete example — webcam edge detector
**Intuition:** The same CV pipeline can run continuously on camera frames.

**Definition:** Capture → grayscale → Gaussian blur → Canny → display.

**Example:**

In [21]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150)

    cv2.imshow('Edges', edges)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## 18. Common mistakes
1. Forgetting `cap.release()`
2. Forgetting `cv2.destroyAllWindows()`
3. Not checking `ret`
4. Processing unnecessarily large frames
5. Assuming requested camera resolution is guaranteed
6. Forgetting OpenCV frames use BGR

## 19. Cheat sheet
| Task | Function |
|---|---|
| Open camera/video | `cv2.VideoCapture()` |
| Read frame | `cap.read()` |
| Check opened | `cap.isOpened()` |
| Get property | `cap.get()` |
| Set property | `cap.set()` |
| Show frame | `cv2.imshow()` |
| Keyboard input | `cv2.waitKey()` |
| Save video | `cv2.VideoWriter()` |
| Release capture | `cap.release()` |
| Close windows | `cv2.destroyAllWindows()` |